In [ ]:
!pip install z3-solver

In [ ]:
from z3 import *

pcb1_width = 20
pcb1_height = 20
pcb2_width = 22
pcb2_height = 27


In [ ]:
from IPython.utils.text import string
class Chip:
    def __init__(self, width: int, height: int, value: bool):
        self.width = width
        self.height = height
        self.value = value

chips = [
    Chip(5, 6, True),
    Chip(5, 6, True),
    Chip(4, 6, False),
    Chip(4, 10, False),
    Chip(5, 7, False),
    Chip(3, 7, True),
    Chip(7, 7, False),
    Chip(6, 10, False),
    Chip(6, 12, False),
    Chip(4, 10, False),
    Chip(6, 9, False),
    Chip(5, 11, False),
    Chip(6, 10, False),
    Chip(5, 10, False)
]

class PCB_Object:
  def __init__(self, name: string, x: int, y: int, width: int, height: int, pcb_id: int):
    self.name = name
    self.x = x
    self.y = y
    self.width = width
    self.height = height
    self.pcb_id = pcb_id


Comment out which exercise you want to run

In [ ]:
pcb_objects = [
    PCB_Object("Battery", 0, 0, 12, 8, 1),
    PCB_Object("Battery", 0, 0, 14, 12, 2),

    # Ex A
    PCB_Object("Camera", 15, 22, 7, 5, 2),

    # Ex B
    #PCB_Object("Camera", 15 - 4, 22, 7 + 4, 5, 2),

    # Ex C
    #PCB_Object("Camera", 15 - 4*3, 22, 7 + 4*3, 5, 2),
]

In [ ]:
X = [ Int('X_%i' % (i + 1)) for i in range(14) ]
Y = [ Int('Y_%i' % (i + 1)) for i in range(14) ]
R = [ Int('R_%i' % (i + 1)) for i in range(14) ]
PCB = [ Int('PCB_%i' % (i + 1)) for i in range(14) ]

val_x = [ And(0 <= X[i], X[i] <= 20) for i in range(14) ]
val_y = [ And(0 <= Y[i], Y[i] <= 20) for i in range(14) ]
val_r = [ And(0 <= R[i], R[i] <= 1) for i in range(14) ]
val_pcb = [ And(1 <= PCB[i], PCB[i] <= 2) for i in range(14) ]

def check_overlap(i, j):
    chip1 = chips[i]
    chip2 = chips[j]

    x1_min = X[i]
    x1_max = If(R[i] == 0, X[i] + chip1.width, X[i] + chip1.height)
    y1_min = Y[i]
    y1_max = If(R[i] == 0, Y[i] + chip1.height, Y[i] + chip1.width)

    x2_min = X[j]
    x2_max = If(R[j] == 0, X[j] + chip2.width, X[j] + chip2.height)
    y2_min = Y[j]
    y2_max = If(R[j] == 0, Y[j] + chip2.height, Y[j] + chip2.width)

    overlap_x_interval = And(x1_min < x2_max, x2_min < x1_max)
    overlap_y_interval = And(y1_min < y2_max, y2_min < y1_max)
    on_same_pcb = PCB[i] == PCB[j]

    return And(overlap_x_interval, overlap_y_interval, on_same_pcb)

no_overlap = [Not(check_overlap(i, j)) for i in range(14) for j in range(14) if i != j]

def check_inside_bounds (i):
  chip = chips[i]

  x_max = If(R[i] == 0, X[i] + chip.width, X[i] + chip.height)
  y_max = If(R[i] == 0, Y[i] + chip.height, Y[i] + chip.width)
  return And(
      If(PCB[i] == 1, x_max <= pcb1_width, x_max <= pcb2_width),
      If(PCB[i] == 1, y_max <= pcb1_height, y_max <= pcb2_height)
    )

bound_constraint = [check_inside_bounds(i) for i in range(14)]

def check_overlap_with_pcb_objects(i, j):
  chip = chips[i]
  pcb_object = pcb_objects[j]

  x1_min = X[i]
  x1_max = If(R[i] == 0, X[i] + chip.width, X[i] + chip.height)
  y1_min = Y[i]
  y1_max = If(R[i] == 0, Y[i] + chip.height, Y[i] + chip.width)

  x2_min = pcb_object.x
  x2_max = pcb_object.x + pcb_object.width
  y2_min = pcb_object.y
  y2_max = pcb_object.y + pcb_object.height

  overlap_x_interval = And(x1_min < x2_max, x2_min < x1_max)
  overlap_y_interval = And(y1_min < y2_max, y2_min < y1_max)
  on_same_pcb = PCB[i] == pcb_object.pcb_id

  return And(overlap_x_interval, overlap_y_interval, on_same_pcb)

overlap_with_object_constraint = [Not(check_overlap_with_pcb_objects(i, j)) for i in range(14) for j in range(len(pcb_objects))]

def is_overheating (i, j):
  chip1 = chips[i]
  chip2 = chips[j]

  x1_min = X[i]
  x1_max = If(R[i] == 0, X[i] + chip1.width, X[i] + chip1.height)
  x1_center = (x1_max + x1_min)
  y1_min = Y[i]
  y1_max = If(R[i] == 0, Y[i] + chip1.height, Y[i] + chip1.width)
  y1_center = (y1_max + y1_min)

  x2_min = X[j]
  x2_max = If(R[j] == 0, X[j] + chip2.width, X[j] + chip2.height)
  x2_center = (x2_max + x2_min)
  y2_min = Y[j]
  y2_max = If(R[j] == 0, Y[j] + chip2.height, Y[j] + chip2.width)
  y2_center = (y2_max + y2_min)

  x_dist = abs(x1_center - x2_center)
  y_dist = abs(y1_center - y2_center)

  x_dist_less_than_20 = x_dist < 20
  y_dist_less_than_20 = y_dist < 20

  return And(x_dist_less_than_20, y_dist_less_than_20, chip1.value == 1, chip2.value == 1, PCB[i] == PCB[j])

overheating_constraint = [Not(is_overheating(i, j)) for i in range(14) for j in range(14) if i != j]


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

def print_chips (model):
    for i in range(14):
      chip = chips[i]
      x = model[X[i]].as_long()
      y = model[Y[i]].as_long()
      width = chip.width
      height = chip.height
      pcb_id = model[PCB[i]].as_long()

      print(f"Chip {i + 1}: x={x}, y={y}, width={width}, height={height}, value={chip.value}, pcb={pcb_id}")

def print_pcb_objects ():
  for i in range(len(pcb_objects)):
    x = pcb_objects[i].x
    y = pcb_objects[i].y
    width = pcb_objects[i].width
    height = pcb_objects[i].height
    name = pcb_objects[i].name
    pcb_id = pcb_objects[i].pcb_id

    print(f"Object '{name}': x={x}, y={y}, width={width}, height={height}, pcb={pcb_id}")

def print_results (model, pcb):
  fig, ax = plt.subplots()

  ax.set_title("PCB_" + str(pcb))

  for i in range(len(pcb_objects)):
    x = pcb_objects[i].x
    y = pcb_objects[i].y
    width = pcb_objects[i].width
    height = pcb_objects[i].height
    name = pcb_objects[i].name
    pcb_id = pcb_objects[i].pcb_id

    if pcb_id != pcb:
      continue

    ax.text(
        x + width / 2,
        y + height / 2,
        name,
        ha="center",
        va="center"
    )

    rect = Rectangle(
        (x, y),
        width, height,
        fill=True,
        facecolor='green',
        alpha=0.5,
        edgecolor="black",
        linewidth=2
    )
    ax.add_patch(rect)

  for i in range(14):
    chip = chips[i]
    x = model[X[i]].as_long()
    y = model[Y[i]].as_long()
    width = chip.width
    height = chip.height
    pcb_id = model[PCB[i]].as_long()

    if pcb_id != pcb:
      continue

    if model[R[i]].as_long() == 1:
        width, height = height, width

    ax.text(
        x + width / 2,
        y + height / 2,
        f"Chip {i + 1}",
        ha="center",
        va="center"
    )

    rect = Rectangle(
        (x, y),
        width, height,
        fill=True,
        facecolor='red' if chip.value else 'blue',
        alpha=0.5,
        edgecolor="black",
        linewidth=2
    )
    ax.add_patch(rect)

  if pcb == 1:
    ax.set_xlim(0, pcb1_width)
    ax.set_ylim(0, pcb1_height)

  if pcb == 2:
    ax.set_xlim(0, pcb2_width)
    ax.set_ylim(0, pcb2_height)

  ax.set_aspect("equal")

  plt.show()


In [ ]:
s = Solver()
s.add(no_overlap + bound_constraint + overlap_with_object_constraint + overheating_constraint + val_x + val_y + val_r + val_pcb)

if s.check() == sat:
    model = s.model()

    print_chips(model)
    print_pcb_objects ()
    print_results(model, 1)
    print_results(model, 2)
else:
    print("No solution!")